In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Set random seed for reproducibility
np.random.seed(419414)

## Task 1

In [ ]:
df = pd.read_csv('pokemon.csv')  # Read in from CSV

print("Dataset shape:", df.shape)
df.head()  # Display the first few rows

In [ ]:
df.info()  # Useful for checking non-null values and data types

In [ ]:
df.isnull().sum()  # Check for missing values across all columns

In [ ]:
sns.countplot(data=df, x='Legendary')
plt.title("Class Distribution of Legendary")
plt.show()

In [ ]:
features = ['Total', 'HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed']

plt.figure(figsize=(16, 10))
for i, col in enumerate(features, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(data=df, x='Legendary', y=col)
    plt.title(f"{col} by Legendary")
plt.tight_layout()
plt.show()

### EDA Summary
The dataset contains 800 Pokémon records with a heavily imbalanced class distribution — only 65 Pokémon (8.1%) are Legendary, compared to 735 non-Legendary. The only column with missing values is `Type_2`, which has 386 missing entries; however, this is expected since many Pokémon only have a single type and this column will be dropped during preprocessing. From the boxplots, `Total`, `Sp_Atk`, `Sp_Def`, and `Speed` show the strongest separation between Legendary and non-Legendary Pokémon, with Legendary Pokémon consistently having much higher median values. `HP`, `Attack`, and `Defense` also show some separation but with more overlap, making them less individually discriminative. Overall, the dataset is clean for the numeric stat columns and the clear differences in base stats suggest that classification should be achievable with reasonable accuracy, though the class imbalance means accuracy alone will not be a sufficient evaluation metric.

## Task 2

In [ ]:
# Drop columns not suitable as features
# ID: arbitrary identifier with no predictive value
# Name: free-text string identifier, not a numeric feature
# Type_1, Type_2: categorical string columns with many categories and 386 missing Type_2 values;
#   encoding them would add high dimensionality for limited benefit given the strong signal in numeric stats

df_model = df.drop(columns=['ID', 'Name', 'Type_1', 'Type_2'])

df_model.head()

### Data Preparation Justification
Four columns were dropped before modelling. `ID` and `Name` are unique identifiers that carry no predictive signal and would cause the model to memorise rather than generalise. `Type_1` and `Type_2` are categorical string columns — `Type_2` also has 386 missing values — and encoding them would add significant dimensionality without a clear benefit, given that the six numeric combat stats and `Total` already provide strong separation between the two classes. The remaining features (`HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`, `Total`, `Generation`) are all numeric and require no further imputation.

In [ ]:
# Separate features and target
X = df_model.drop('Legendary', axis=1)
y = df_model['Legendary'].astype(int)

student_id = 419414  # <-- replace with your actual student ID

# First split: train (60%) vs temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=student_id
)

# Second split: validation (20%) and test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=student_id
)

# Confirm shapes
X_train.shape, X_val.shape, X_test.shape

In [ ]:
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### Model A — Logistic Regression

#### Why Logistic Regression?
Logistic Regression is a sensible baseline for this problem because the numeric stats (Total, Sp_Atk, Speed, etc.) show clear linear separation between Legendary and non-Legendary Pokémon in the boxplots. It is interpretable, computationally efficient, and performs well when features are scaled — which we have already applied. It also provides probabilistic outputs, which is useful for understanding how confidently a prediction is made on imbalanced data.

In [ ]:
log_reg = LogisticRegression(max_iter=1000)

param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"]
}

grid_lr = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

In [ ]:
print("Model A — Logistic Regression")
print("Best Parameters:", grid_lr.best_params_)
print("Mean CV Score:", grid_lr.best_score_)

### Model B — k-Nearest Neighbours

#### Why KNN?
KNN is a non-parametric classifier that makes no assumptions about the underlying distribution of the data. It can capture non-linear boundaries between Legendary and non-Legendary Pokémon, which is useful given that some features like `HP` and `Attack` show less clear linear separation. KNN benefits directly from the feature scaling already applied, and tuning `k` and the distance metric allows it to adapt well to this relatively small dataset.

In [ ]:
knn = KNeighborsClassifier()

param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_knn.fit(X_train_scaled, y_train)

In [ ]:
print("Model B — KNN")
print("Best Parameters:", grid_knn.best_params_)
print("Mean CV Score:", grid_knn.best_score_)

### Model C — Random Forest

#### Why Random Forest?
Random Forest is a strong choice for this dataset because it handles non-linear relationships and naturally models interactions between features such as the combination of `Total`, `Speed`, and `Sp_Atk` that distinguish Legendary Pokémon. It is robust to noise, provides built-in feature importance, and does not require feature scaling. It is also well-suited to imbalanced datasets because averaging across many trees reduces the risk of overfitting to the majority class.

In [ ]:
rf = RandomForestClassifier(random_state=student_id)

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)  # RF does NOT need scaling

In [ ]:
print("Model C — Random Forest")
print("Best Parameters:", grid_rf.best_params_)
print("Mean CV Score:", grid_rf.best_score_)

## Task 4

In [ ]:
# Best estimators
best_lr = grid_lr.best_estimator_
best_knn = grid_knn.best_estimator_
best_rf = grid_rf.best_estimator_

# Predictions
pred_lr = best_lr.predict(X_val_scaled)
pred_knn = best_knn.predict(X_val_scaled)
pred_rf = best_rf.predict(X_val)   # RF does not use scaled data

# Metrics
results = {
    "Model": ["Logistic Regression", "KNN", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_val, pred_lr),
        accuracy_score(y_val, pred_knn),
        accuracy_score(y_val, pred_rf)
    ],
    "F1 Score": [
        f1_score(y_val, pred_lr),
        f1_score(y_val, pred_knn),
        f1_score(y_val, pred_rf)
    ]
}

In [ ]:
results_df = pd.DataFrame(results)
results_df

### Model Comparison Summary
Both KNN and Random Forest outperform Logistic Regression on the validation set, achieving higher accuracy and substantially better F1 scores. Logistic Regression's lower F1 score suggests it struggles to correctly identify Legendary Pokémon, likely because the decision boundary between classes is not perfectly linear across all features. Given the severe class imbalance in this dataset (735 non-Legendary vs. 65 Legendary), F1 score is the more informative metric — a classifier that simply predicted 'non-Legendary' for every sample would achieve over 91% accuracy while identifying zero Legendary Pokémon, so accuracy alone is misleading. The F1 score penalises both false positives and false negatives equally, making it a better reflection of whether the model is actually learning to detect the minority class. KNN and Random Forest both achieve comparable F1 scores on the validation set, making either a reasonable choice as the final model.

## Task 5
### Final Model Selection
I am selecting the Random Forest classifier as my final model. Although KNN achieves the same validation accuracy and F1 score, Random Forest is more robust to noise and generalises better on unseen data because it averages predictions across many decorrelated trees rather than relying on proximity to training samples. Its ability to model feature interactions and handle class imbalance without requiring feature scaling also makes it the more principled choice for this problem.

In [ ]:
# Combine training + validation sets
X_train_final = np.vstack([X_train, X_val])
y_train_final = np.hstack([y_train, y_val])

# Retrain Random Forest with best hyperparameters
best_rf_params = grid_rf.best_params_

final_rf = RandomForestClassifier(
    **best_rf_params,
    random_state=student_id
)

final_rf.fit(X_train_final, y_train_final)

In [ ]:
test_pred = final_rf.predict(X_test)

print("Final Model — Random Forest")
print(classification_report(y_test, test_pred, target_names=["Non-Legendary", "Legendary"]))

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Non-Legendary", "Legendary"],
            yticklabels=["Non-Legendary", "Legendary"])
plt.title("Confusion Matrix — Final Random Forest Model")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

### Final Model Interpretation
The final Random Forest model performs very well on the non-Legendary class, achieving near-perfect precision and recall, but shows a notable drop in recall for the Legendary class — correctly identifying roughly half of the true Legendary Pokémon while missing the rest. This asymmetry is a direct consequence of class imbalance: with only 13 Legendary Pokémon in the test set, even a small number of false negatives significantly reduces recall for that class. The confusion matrix confirms this pattern, showing all non-Legendary Pokémon are classified correctly while several Legendary ones are misclassified as non-Legendary. This suggests the model has learned the majority class very well but has limited exposure to Legendary examples during training. While the model would be useful as a preliminary screening tool, it would not be suitable for deployment in a setting where missing Legendary Pokémon carries a high cost — in such cases, techniques like oversampling (SMOTE), class weighting, or lowering the classification threshold should be explored.